In [1]:
import warnings
from pyspark.sql import SparkSession, Window
import pyspark.sql.functions as F
from pyspark.sql.types import *
import pyspark.sql.dataframe
from datetime import datetime
from copy import deepcopy
from IPython.display import display
import pandas as pd
import sys
from tqdm import tqdm
import os
from pyspark.sql.utils import AnalysisException
import json
#sys.path.append(os.path.abspath(".."))
#from utils.kafka_config import KafkaConfig
import yaml
from pathlib import Path
import uuid
# Configure pandas to show ful output without truncation
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)     # Show all rows
pd.set_option('display.max_colwidth', None) # Don't truncate columns content
pd.set_option('display.width', None)        # Use full width

print(" Libraries imported successfully")
warnings.filterwarnings("ignore")

 Libraries imported successfully


In [2]:
# for juptor only 
sys.path.append(os.path.abspath("../.."))
from ddl.tables_schema import  (
                                 batch_tracker_schema
                                )

In [4]:
from utils.postgres_config import PostgresConfig

POSTGRES_CONFIG = PostgresConfig(config_path='../../conf/batches_mapping.yaml')._get_config_json()
jdbc_url = POSTGRES_CONFIG.get("postgress").get("jdbc_url")
jdbc_properties = POSTGRES_CONFIG.get("postgress").get("jdbc_properties")
jdbc_properties =  {k: v for d in jdbc_properties for k, v in d.items()}
bucket = POSTGRES_CONFIG.get("bucket")
bucket

{'postgress': {'jdbc_url': 'jdbc:postgresql://postgres-source:5432/master_db', 'jdbc_properties': [{'user': 'admin'}, {'password': 'admin'}, {'driver': 'org.postgresql.Driver'}]}, 'bucket': {'bronze-layer': 'lake.bronze'}}


{'bronze-layer': 'lake.bronze'}

In [4]:
bucket_name = "lake.bronze"

In [5]:

def print_as_df(df, limit = 5):
    if isinstance(df, pyspark.sql.dataframe.DataFrame):
         display(df.limit(limit).toPandas())
    else:
        tqdm.write('Unknow types. Just spark dataframe is acceptable')





def get_last_watermark(table_name:str, layer:str, prev_layer:str) -> datetime:
    last_watermark = datetime(2026, 5, 1) # default date
    try: 
        last_watermark = spark.sql(f"""
        SELECT COALESCE(
            (SELECT last_watermark FROM lake.{layer}.watermarks WHERE table_name = '{table_name}'),
            (SELECT MIN(ingestion_ts) FROM lake.{prev_layer}.{table_name})
        ) as last_watermark
    """).collect()[0][0]
        return last_watermark
    except Exception as e : 
        tqdm.write(e, "\ndefault water_mark",last_watermark)
        raise


def flatten_kafka_payload(df:pyspark.sql.dataframe.DataFrame, table_schema):
     #if isinstance(df, pyspark.sql.dataframe.DataFrame):
     return df.withColumn(
        "after_payload_value", from_json(col("after_payload"),table_schema)
                        ).select(
                                "*",
                                "after_payload_value.*",    
                            ).filter(
                                    (col("operation") != 'd') & (col("after_payload").isNotNull() 
                                                                )).drop("after_payload_value", "after_payload", "before_payload")
     
def fetch_bronze_layer_data(table_path_name, water_mark):
    return spark.sql(f""" SELECT * 
                          FROM {table_path_name}
                          WHERE ingestion_ts >= CAST('{water_mark}' AS TIMESTAMP)""")



def update_watermark_table(table_name, layer_path = "lake.silver"):
    try:
        spark.sql(f"""
                    MERGE INTO {layer_path}.watermarks w
                    USING (SELECT '{table_name}' as table_name, CURRENT_TIMESTAMP as last_watermark, CURRENT_TIMESTAMP as updated_at) src
                    ON w.table_name = src.table_name
                    WHEN MATCHED THEN UPDATE SET w.last_watermark = src.last_watermark, w.updated_at = CURRENT_TIMESTAMP
                    WHEN NOT MATCHED THEN INSERT *
                    """)
        tqdm.write(f"Updating the watermark table {layer_path}.{table_name} Successed ")
    except Exception as e:
        raise Exception(f"Could not update the water mark for {table_name}\n", e)

def run_func(func, *args, **kwargs):
    start_time = datetime.now()

    func(*args, **kwargs)

    end_time = datetime.now()

    duration = (end_time - start_time).total_seconds()

    hours, remainder = divmod(duration, 3600)
    minutes, seconds = divmod(remainder, 60)

    tqdm.write(
        f"Duration: {int(hours):02d}:"
        f"{int(minutes):02d}:"
        f"{seconds:06.3f}"
    )

In [6]:
# tables = {
#     "customers":{
#         "PK":"customer_id",
#         "max_id":1,
#         "min_id":1,
#         "count":1,
#         "is_cdc":False
#     },
#     "cities":{
#         "PK":"city_id",
#         "max_id":1,
#         "min_id":1,
#         "count":1,
#         "is_cdc":False
#     },
#     "zones":{
#         "PK":"zone_id",
#         "max_id":1,
#         "min_id":1,
#         "count":1,
#         "is_cdc":False
#     },
#     "restaurants":{
#         "PK":"restaurant_id",
#         "max_id":1,
#         "min_id":1,
#         "count":1,
#         "is_cdc":False
#     },
#     "orders":{
#         "PK":"order_id",
#         "max_id":1,
#         "min_id":1,
#         "count":1,
#         "is_cdc":True
#     },
#     "order_items":{
#         "PK":"order_item_id",
#         "max_id":1,
#         "min_id":1,
#         "count":1,
#         "is_cdc":True
#     },
#     "payments":{
#         "PK":"payment_id",
#         "max_id":1,
#         "min_id":1,
#         "count":1,
#         "is_cdc":True
#     },
#     "order_status_events":{
#         "PK":"event_id",
#         "max_id":1,
#         "min_id":1,
#         "count":1,
#         "is_cdc":True
#     },
#     "reviews":{
#         "PK":"review_id",
#         "max_id":1,
#         "min_id":1,
#         "count":1,
#         "is_cdc":True
#     },
#     "drivers":{
#         "PK":"driver_id",
#         "max_id":1,
#         "min_id":1,
#         "count":1,
#         "is_cdc":True
#     },
#     "menu_items":{
#         "PK":"menu_item_id",
#         "max_id":1,
#         "min_id":1,
#         "count":1,
#         "is_cdc":True
#     },
# }

In [7]:
import json
tables = dict()
with open("tables.json", "r") as f:
    tables = json.load(f)

print(tables)

{'customers': {'PK': 'customer_id', 'max_id': 1, 'min_id': 1, 'count': 1, 'is_cdc': False}, 'cities': {'PK': 'city_id', 'max_id': 1, 'min_id': 1, 'count': 1, 'is_cdc': False}, 'zones': {'PK': 'zone_id', 'max_id': 1, 'min_id': 1, 'count': 1, 'is_cdc': False}, 'restaurants': {'PK': 'restaurant_id', 'max_id': 1, 'min_id': 1, 'count': 1, 'is_cdc': False}, 'orders': {'PK': 'order_id', 'max_id': 1, 'min_id': 1, 'count': 1, 'is_cdc': True}, 'order_items': {'PK': 'order_item_id', 'max_id': 1, 'min_id': 1, 'count': 1, 'is_cdc': True}, 'payments': {'PK': 'payment_id', 'max_id': 1, 'min_id': 1, 'count': 1, 'is_cdc': True}, 'order_status_events': {'PK': 'event_id', 'max_id': 1, 'min_id': 1, 'count': 1, 'is_cdc': True}, 'reviews': {'PK': 'review_id', 'max_id': 1, 'min_id': 1, 'count': 1, 'is_cdc': True}, 'drivers': {'PK': 'driver_id', 'max_id': 1, 'min_id': 1, 'count': 1, 'is_cdc': True}, 'menu_items': {'PK': 'menu_item_id', 'max_id': 1, 'min_id': 1, 'count': 1, 'is_cdc': True}}


In [8]:
try: 
    from pyspark import SparkContext
    sc = SparkContext._active_spark_context
    if sc: 
        sc.stop()
        print(" Stoped previous SparkContext")
except:
    pass
    
spark = (
    SparkSession
    .builder
    .appName("Transform data from bornze to silve, Batches")
    .config("spark.streaming.stopGracefullyOnShutdown", True)
    .config("spark.sql.shuffle.partitions", 4)
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") 
    .config("spark.sql.catalog.lake",
            "org.apache.iceberg.spark.SparkCatalog") 
    .config("spark.sql.catalog.lake.type", "hadoop") 
    .config("spark.sql.catalog.lake.warehouse", "s3a://lake/warehouse") 
    .config("spark.ssl.enabled", "false")
    .master("local[*]")
    .getOrCreate()
)
spark


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/05 16:54:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [11]:
spark.sql("SELECT * FROM lake.bronze.batches_tracker").collect()

26/07/05 16:55:24 WARN InstanceMetadataServiceResourceFetcher: Fail to retrieve token 
com.amazonaws.SdkClientException: Failed to connect to service endpoint: 
	at com.amazonaws.internal.EC2ResourceFetcher.doReadResource(EC2ResourceFetcher.java:100)
	at com.amazonaws.internal.InstanceMetadataServiceResourceFetcher.getToken(InstanceMetadataServiceResourceFetcher.java:91)
	at com.amazonaws.internal.InstanceMetadataServiceResourceFetcher.readResource(InstanceMetadataServiceResourceFetcher.java:69)
	at com.amazonaws.internal.EC2ResourceFetcher.readResource(EC2ResourceFetcher.java:66)
	at com.amazonaws.auth.InstanceMetadataServiceCredentialsFetcher.getCredentialsEndpoint(InstanceMetadataServiceCredentialsFetcher.java:60)
	at com.amazonaws.auth.InstanceMetadataServiceCredentialsFetcher.getCredentialsResponse(InstanceMetadataServiceCredentialsFetcher.java:48)
	at com.amazonaws.auth.BaseCredentialsFetcher.fetchCredentials(BaseCredentialsFetcher.java:124)
	at com.amazonaws.auth.BaseCredentials

Py4JJavaError: An error occurred while calling o39.sql.
: org.apache.iceberg.exceptions.RuntimeIOException: Failed to refresh the table
	at org.apache.iceberg.hadoop.HadoopTableOperations.refresh(HadoopTableOperations.java:128)
	at org.apache.iceberg.hadoop.HadoopTableOperations.current(HadoopTableOperations.java:86)
	at org.apache.iceberg.BaseMetastoreCatalog.loadTable(BaseMetastoreCatalog.java:49)
	at org.apache.iceberg.shaded.com.github.benmanes.caffeine.cache.BoundedLocalCache.lambda$doComputeIfAbsent$14(BoundedLocalCache.java:2406)
	at java.base/java.util.concurrent.ConcurrentHashMap.compute(Unknown Source)
	at org.apache.iceberg.shaded.com.github.benmanes.caffeine.cache.BoundedLocalCache.doComputeIfAbsent(BoundedLocalCache.java:2404)
	at org.apache.iceberg.shaded.com.github.benmanes.caffeine.cache.BoundedLocalCache.computeIfAbsent(BoundedLocalCache.java:2387)
	at org.apache.iceberg.shaded.com.github.benmanes.caffeine.cache.LocalCache.computeIfAbsent(LocalCache.java:108)
	at org.a

# Connect to postgres source

In [642]:
# jdbc_properties = [{
#     "user": "admin",
#     "password": "admin",
#     "driver": "org.postgresql.Driver"
# }]
# jdbc_properties[0]

In [15]:
#jdbc_url = "jdbc:postgresql://postgres-source:5432/master_db"

# jdbc_properties = [{
#     "user": "admin",
#     "password": "admin",
#     "driver": "org.postgresql.Driver"
# }]
def read_postgres_table(table_name, sql_query):
 
    return spark.read \
        .format("jdbc") \
        .option("url", jdbc_url) \
        .options(**jdbc_properties) \
        .option("query", sql_query) \
        .load()
def read_postgres_table_parallel(sql_query, partition_column, lower, upper, num_partitions=4):
    return spark.read \
        .format("jdbc") \
        .option("url", jdbc_url) \
        .option("dbtable", sql_query) \
        .option("partitionColumn", partition_column) \
        .option("lowerBound", lower) \
        .option("upperBound", upper) \
        .option("numPartitions", num_partitions) \
        .options(**jdbc_properties) \
        .load()

# Prepare metadata for tables

In [16]:
def get_metadata_tables(tables:dict, initial_load = False, is_initial_load_with_cdc = False):
    """
    Retrieve metadata required for batch generation from the source tables.

    This function collects the total number of records and the ID boundaries
    (minimum and maximum primary key values) for each table. The collected
    metadata is later used to generate extraction batches.

    Depending on the loading strategy, the function supports both initial
    and incremental loads:

    - Initial Load:
        Retrieves metadata for the entire table.

    - Incremental Load:
        Retrieves metadata only for records updated after the last generated
        batch timestamp stored in the batch tracker table.

    The function also handles CDC (Change Data Capture) tables according to
    the provided configuration flags.

    Workflow:
    1. Create a copy of the input tables dictionary.
    2. Optionally retrieve the last generated batch timestamps.
    3. Iterate through all configured tables.
    4. Skip or include CDC tables based on the load configuration.
    5. Build the appropriate metadata query.
    6. Retrieve count, maximum ID, and minimum ID from the source.
    7. Remove tables with no new records.
    8. Store the collected metadata inside the table configuration.
    9. Return the updated tables dictionary.

    Args:
        tables (dict):
            Dictionary containing table configurations.
            Example:
            {
                "customers": {
                    "PK": "customer_id",
                    "is_cdc": False
                }
            }

        initial_load (bool, optional):
            Determines the loading strategy.

            - True: Read metadata for the entire source table.
            - False: Read metadata only for newly updated records.

            Default is False.

        is_initial_load_with_cdc (bool, optional):
            Controls whether CDC tables should participate in the initial load.

            - True: Include CDC tables during the initial load.
            - False: Exclude CDC tables.

            Default is False.

    Returns:
        dict:
            Updated tables dictionary containing:

            - count
            - max_id
            - min_id

            Example:
            {
                "customers": {
                    "PK": "customer_id",
                    "is_cdc": False,
                    "count": 500000,
                    "max_id": 500000,
                    "min_id": 1
                }
            }

    Raises:
        ValueError:
            If an incremental load is requested but no previous successful
            batch generation exists in the batch tracker table.

    Note:
        - Incremental loading assumes that source tables contain an
          `updated_at` column.
        - CDC tables are automatically excluded unless explicitly enabled
          for the initial load.
        - Tables with no new or updated records are removed from the
          returned dictionary.
    """
    tables = deepcopy(tables)
    last_generated_batches = spark.createDataFrame([], "job_name STRING, generated_at_ts TIMESTAMP")
    def get_metadata_query(t, table_pk, initial_load, last_generated_batches = spark.createDataFrame([], "job_name STRING, generated_at_ts TIMESTAMP")):
        
        if initial_load:
            tqdm.write("technique type: inital load")

            return  f"""
                    SELECT count(*), max({table_pk}), min({table_pk})  
                    FROM {t} 
                    """
        else:
            tqdm.write("technique type: incremental load")
            
            last_generated_batch = (
                                        last_generated_batches
                                        .filter(F.col("job_name").contains("customers"))
                                        .agg(F.max("generated_at_ts").alias("generated_at_ts"))
                                    ).collect()[0][0]
            if not last_generated_batch:
                raise ValueError(f"The last_generated_batch value is {last_generated_batch} which is means the batch tracker table mostly is empty no inital load occrs before so that can't start incremental load technique, you must start inital load first.")
            write.tqdm(f"Last generated batch timestamp: {last_generated_batch}")
            return  f"""
                    SELECT count(*), max({table_pk}), min({table_pk})  
                    FROM {t} 
                    WHERE updated_at >= '{last_generated_batch}'
                    """
    pbar = tqdm(tables)
    # is_cdc:false and is_initial_load_with_cdc:true --> no effect 
    # is_cdc:false and is_initial_load_with_cdc:false --> no effect 
    # is_cdc:true and initial_load:true and is_initial_load_with_cdc:false --> just inital load without cdc tables
    # is_cdc:true and initial_load:false and is_initial_load_with_cdc:true --> inital load for all tables
    pbar.set_description(f"Get Metadata Tables")
    for idx, t in enumerate(list(pbar)):
        if tables.get(t).get("is_cdc"):
            # skip the cdc tables type which usualy dose not have update_date_ts column and drop it from the object
            if not (is_initial_load_with_cdc and initial_load):
                del tables[t]
                tqdm.write(f"table {t} is cdc, reomved from the dict")
                continue 
        if not initial_load:
            last_generated_batches = spark.sql(f""" 
                                        SELECT job_name, max(generated_at_ts) as generated_at_ts
                                        FROM lake.bronze.batches_tracker
                                        GROUP BY job_name
                        """).cache()
        
        table_pk = tables.get(t).get('PK')
        
        table_metadata_q = get_metadata_query(t = t, 
                                              table_pk = table_pk, 
                                              initial_load = initial_load, last_generated_batches= last_generated_batches)
        print(f"table name:{t}\n table PK:{table_pk}")
        df = read_postgres_table(t, table_metadata_q)

        count = df.collect()[0][1]
        max_id = df.collect()[0][1]
        min_id = df.collect()[0][2]

        if not count:
            tqdm.write(f" No update occur in source. so will be skiped and removed from dict\n")
            del tables[t] 
            continue
        tables[t]['max_id'] = max_id
        tables[t]['min_id'] = min_id
        tables[t]['count'] = count
        tqdm.write(f"{idx}, table:{t}, details:{tables.get(t)}\n") 
        tqdm.write("==================================================================================")
    if last_generated_batches:
        last_generated_batches.unpersist()
    return tables

In [17]:
new_tables = get_metadata_tables(tables, initial_load=True, is_initial_load_with_cdc= True)
new_tables

Get Metadata Tables: 100%|██████████| 11/11 [00:00<00:00, 17945.29it/s]


technique type: inital load
table name:customers
 table PK:customer_id


Py4JJavaError: An error occurred while calling o133.load.
: org.postgresql.util.PSQLException: The connection attempt failed.
	at org.postgresql.core.v3.ConnectionFactoryImpl.openConnectionImpl(ConnectionFactoryImpl.java:354)
	at org.postgresql.core.ConnectionFactory.openConnection(ConnectionFactory.java:54)
	at org.postgresql.jdbc.PgConnection.<init>(PgConnection.java:263)
	at org.postgresql.Driver.makeConnection(Driver.java:443)
	at org.postgresql.Driver.connect(Driver.java:297)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.BasicConnectionProvider.getConnection(BasicConnectionProvider.scala:49)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.ConnectionProviderBase.create(ConnectionProvider.scala:102)
	at org.apache.spark.sql.jdbc.JdbcDialect.$anonfun$createConnectionFactory$1(JdbcDialects.scala:160)
	at org.apache.spark.sql.jdbc.JdbcDialect.$anonfun$createConnectionFactory$1$adapted(JdbcDialects.scala:156)
	at org.apache.spark.sql.execution.datasources

In [26]:
tables_with_batches = generate_batches(new_tables)
tables_with_batches

Generate Batches: 100%|██████████| 11/11 [00:00<00:00, 345.40it/s, table=menu_items]

customers: batch_size_per_table 41800, count: 41800
cities: batch_size_per_table 5, count: 5
zones: batch_size_per_table 61, count: 61
restaurants: batch_size_per_table 3000, count: 3000
orders: batch_size_per_table 100000, count: 2096889
order_items: batch_size_per_table 100000, count: 6188318
payments: batch_size_per_table 100000, count: 2096889
order_status_events: batch_size_per_table 100000, count: 3427625
reviews: batch_size_per_table 100000, count: 139481
drivers: batch_size_per_table 15000, count: 15000
menu_items: batch_size_per_table 22794, count: 22794


{'customers': {'PK': 'customer_id',
  'max_id': 41800,
  'min_id': 1,
  'count': 41800,
  'is_cdc': False,
  'new_batches': [(0, 1, 41800)]},
 'cities': {'PK': 'city_id',
  'max_id': 5,
  'min_id': 1,
  'count': 5,
  'is_cdc': False,
  'new_batches': [(0, 1, 5)]},
 'zones': {'PK': 'zone_id',
  'max_id': 61,
  'min_id': 1,
  'count': 61,
  'is_cdc': False,
  'new_batches': [(0, 1, 61)]},
 'restaurants': {'PK': 'restaurant_id',
  'max_id': 3000,
  'min_id': 1,
  'count': 3000,
  'is_cdc': False,
  'new_batches': [(0, 1, 3000)]},
 'orders': {'PK': 'order_id',
  'max_id': 2096889,
  'min_id': 1,
  'count': 2096889,
  'is_cdc': True,
  'new_batches': [(0, 1, 100000),
   (1, 100001, 200000),
   (2, 200001, 300000),
   (3, 300001, 400000),
   (4, 400001, 500000),
   (5, 500001, 600000),
   (6, 600001, 700000),
   (7, 700001, 800000),
   (8, 800001, 900000),
   (9, 900001, 1000000),
   (10, 1000001, 1100000),
   (11, 1100001, 1200000),
   (12, 1200001, 1300000),
   (13, 1300001, 1400000),
   (

In [63]:
newb = insert_new_batches(tables_with_batches, load_type = 'incremntial_load', tolerate_with_duplicate=False)
newb

select * from lake.bronze.batches_tracker where  status in ('pending', 'failed' , 'success' )


Inserting New Batches:   9%|▉         | 1/11 [00:02<00:25,  2.56s/it, job_name=cities_incremntial_load, table=cities]      

this batch is exist in prev generateing for table customers, batch_number 0 and there status: [Row(status='success'), Row(status='pending')]


Inserting New Batches:  18%|█▊        | 2/11 [00:02<00:10,  1.21s/it, job_name=zones_incremntial_load, table=zones]  

this batch is exist in prev generateing for table cities, batch_number 0 and there status: [Row(status='success'), Row(status='pending')]


...

this batch is exist in prev generateing for table reviews, batch_number 1 and there status: [Row(status='success'), Row(status='pending')]


Inserting New Batches:  91%|█████████ | 10/11 [00:34<00:03,  3.03s/it, job_name=menu_items_incremntial_load, table=menu_items]

this batch is exist in prev generateing for table drivers, batch_number 0 and there status: [Row(status='success'), Row(status='pending')]


Inserting New Batches: 100%|██████████| 11/11 [00:34<00:00,  3.13s/it, job_name=menu_items_incremntial_load, table=menu_items]


this batch is exist in prev generateing for table menu_items, batch_number 0 and there status: [Row(status='success'), Row(status='pending')]
inserting successed


In [60]:
newb

In [61]:
print_as_df(spark.sql("select * from lake.bronze.batches_tracker where "))

,id,job_name,batch_number,lower_bound,upper_bound,status,rows_processed,started_at_ts,completed_at_ts,error_message,generated_at_ts
0,aadc1a48-06d7-465b-af4b-fcc0cafcf9a2,order_status_events_inital_load,0,1,100000,success,100000,2026-06-10 02:38:13.495022,2026-06-10 02:38:19.944802,None,2026-06-10 01:58:24.850389
1,825a3485-4dba-45ba-a049-39502ffdaeea,order_status_events_inital_load,1,100001,200000,success,100000,2026-06-10 02:38:24.653549,2026-06-10 02:38:30.908837,None,2026-06-10 01:58:24.876575
2,71f6ce42-be97-471a-a0b7-702995f8391f,order_status_events_inital_load,2,200001,300000,success,100000,2026-06-10 02:38:35.132096,2026-06-10 02:38:42.722509,None,2026-06-10 01:58:24.905260
3,0cfa67b8-3a61-4ca0-83d9-209e71cf241e,order_status_events_inital_load,3,300001,400000,success,100000,2026-06-10 02:38:47.159934,2026-06-10 02:38:52.830735,None,2026-06-10 01:58:24.933764
4,08d55d9c-390e-425a-9e09-38127b67dba0,order_status_events_inital_load,4,400001,500000,success,100000,2026-06-10 02:38:56.559416,2026-06-10 02:39:04.938473,None,2026-06-10 01:58:24.958576


 ## Step 1 — Create Batch Tracker Table

In [115]:
                    
spark.sql(""" 
        CREATE OR REPLACE TABLE  lake.bronze.batches_tracker (
             id STRING,
             job_name STRING,
             batch_number INTEGER,
             lower_bound INTEGER,
             upper_bound INTEGER,
             status STRING,
             rows_processed BIGINT,
             started_at_ts TIMESTAMP,
             completed_at_ts TIMESTAMP,
             error_message STRING,
             generated_at_ts TIMESTAMP 
         )
         USING iceberg
         PARTITIONED BY (job_name)
         TBLPROPERTIES (
        'format-version'                  = '2',
        'write.format.default'            = 'parquet',
        'write.parquet.compression-codec' = 'zstd',
        'write.target-file-size-bytes'    = '134217728'
        )
          """)

DataFrame[]

In [116]:
print_as_df(spark.sql("select * from lake.bronze.batches_tracker;"))

,id,job_name,batch_number,lower_bound,upper_bound,status,rows_processed,started_at_ts,completed_at_ts,error_message,generated_at_ts


## Step 2 — Get Bounds Dynamically

## Step 3 — Generate Batches

In [20]:
def generate_batches(tables, batch_size =100_000):
    """
    Generate ID-based batches for each table.

    This function divides the records of each table into smaller chunks
    based on the specified batch size. The generated batches are stored
    in the `new_batches` key of each table configuration.

    Each batch is represented as a tuple:

        (batch_number, lower_bound, upper_bound)

    Workflow:
    1. Iterate through all tables.
    2. Determine the effective batch size for each table.
    3. Split the ID range into multiple batches.
    4. Store the generated batches inside `tables[table]["new_batches"]`.
    5. Return the updated tables dictionary.

    Args:
        tables (dict):
            Dictionary containing table metadata.
            Expected format:
            {
                "customers": {
                    "count": 1000000,
                    "min_id": 1
                }
            }

        batch_size (int, optional):
            Maximum number of records per batch.
            Default is 100,000.

    Returns:
        dict:
            The updated tables dictionary with a `new_batches`
            key added to each table.

            Example:
            {
                "customers": {
                    "count": 250000,
                    "min_id": 1,
                    "new_batches": [
                        (0, 1, 100000),
                        (1, 100001, 200000),
                        (2, 200001, 250000)
                    ]
                }
            }

    Note:
        - If the total record count is smaller than the batch size,
          a single batch is generated.
        - The last batch may contain fewer records than the specified
          batch size.
    """
    pbar = tqdm(tables, desc="Generate Batches", leave=True)
    for t in pbar:
        pbar.set_postfix(table = t)
        tables[t]["new_batches"] = list()
        count = tables.get(t).get("count")
        batch_size_per_table = min(batch_size, count)
        min_id = tables.get(t).get("min_id")
        tqdm.write(f"{t}: batch_size_per_table {batch_size_per_table}, count: {count}")
        for idx, a in enumerate(range(min_id, count, batch_size_per_table)):
            
            if  count > batch_size_per_table+a-1:
                tp = (idx, a, batch_size_per_table+a-1)
         
                tables[t]["new_batches"].append(tp)
            else:
                tp = (idx, a, count)
            
                tables[t]["new_batches"].append(tp)
    return tables

In [48]:
generate_batches(tables)

Generate Batches: 100%|██████████| 11/11 [00:00<00:00, 352.14it/s, table=menu_items]

customers: batch_size_per_table 1, count: 1
cities: batch_size_per_table 1, count: 1
zones: batch_size_per_table 1, count: 1
restaurants: batch_size_per_table 1, count: 1
orders: batch_size_per_table 1, count: 1
order_items: batch_size_per_table 1, count: 1
payments: batch_size_per_table 1, count: 1
order_status_events: batch_size_per_table 1, count: 1
reviews: batch_size_per_table 1, count: 1
drivers: batch_size_per_table 1, count: 1
menu_items: batch_size_per_table 1, count: 1


{'customers': {'PK': 'customer_id',
  'max_id': 1,
  'min_id': 1,
  'count': 1,
  'is_cdc': False,
  'new_batches': []},
 'cities': {'PK': 'city_id',
  'max_id': 1,
  'min_id': 1,
  'count': 1,
  'is_cdc': False,
  'new_batches': []},
 'zones': {'PK': 'zone_id',
  'max_id': 1,
  'min_id': 1,
  'count': 1,
  'is_cdc': False,
  'new_batches': []},
 'restaurants': {'PK': 'restaurant_id',
  'max_id': 1,
  'min_id': 1,
  'count': 1,
  'is_cdc': False,
  'new_batches': []},
 'orders': {'PK': 'order_id',
  'max_id': 1,
  'min_id': 1,
  'count': 1,
  'is_cdc': True,
  'new_batches': []},
 'order_items': {'PK': 'order_item_id',
  'max_id': 1,
  'min_id': 1,
  'count': 1,
  'is_cdc': True,
  'new_batches': []},
 'payments': {'PK': 'payment_id',
  'max_id': 1,
  'min_id': 1,
  'count': 1,
  'is_cdc': True,
  'new_batches': []},
 'order_status_events': {'PK': 'event_id',
  'max_id': 1,
  'min_id': 1,
  'count': 1,
  'is_cdc': True,
  'new_batches': []},
 'reviews': {'PK': 'review_id',
  'max_id': 

In [341]:
tables

{'customers': {'PK': 'customer_id',
  'max_id': None,
  'min_id': None,
  'count': None,
  'is_cdc': False,
  'new_batches': []},
 'cities': {'PK': 'city_id',
  'max_id': None,
  'min_id': None,
  'count': None,
  'is_cdc': False},
 'zones': {'PK': 'zone_id',
  'max_id': None,
  'min_id': None,
  'count': None,
  'is_cdc': False},
 'restaurants': {'PK': 'restaurant_id',
  'max_id': None,
  'min_id': None,
  'count': None,
  'is_cdc': False},
 'orders': {'PK': 'order_id',
  'max_id': 1,
  'min_id': 1,
  'count': 1,
  'is_cdc': True},
 'order_items': {'PK': 'order_item_id',
  'max_id': 1,
  'min_id': 1,
  'count': 1,
  'is_cdc': True},
 'payments': {'PK': 'payment_id',
  'max_id': 1,
  'min_id': 1,
  'count': 1,
  'is_cdc': True},
 'order_status_events': {'PK': 'event_id',
  'max_id': 1,
  'min_id': 1,
  'count': 1,
  'is_cdc': True},
 'reviews': {'PK': 'review_id',
  'max_id': 1,
  'min_id': 1,
  'count': 1,
  'is_cdc': True},
 'drivers': {'PK': 'driver_id',
  'max_id': 1,
  'min_id': 1

## Step 4 — Register All Batches

In [48]:
print_as_df(spark.sql("select * from lake.bronze.batches_tracker;"))

,id,job_name,batch_number,lower_bound,upper_bound,status,rows_processed,started_at_ts,completed_at_ts,error_message,generated_at_ts


In [49]:
spark.sql("TRUNCATE TABLE lake.bronze.batches_tracker;")

DataFrame[]

In [112]:
# old_batches_q = """select * 
#                     from lake.bronze.batches_tracker 
#                     where  status in ('pending', 'failed', 'success') and job_name = 'order_status_events_inital_load' 
#                      AND lower_bound =1 and upper_bound= 100000 
#                     """
# df_old_batches = spark.sql(old_batches_q).cache().collect()[0]

# #spark.sql(f"select * from lake.bronze.batches_tracker where  status in ('pending', 'failed' )")
# if df_old_batches.status == 'success' and df_old_batches.rows_processed == 100_000:
#     print(df_old_batches)

In [156]:
def insert_new_batches(tables, load_type = 'incremental', tolerate_with_duplicate = False):
    """ 
    Generate and insert new batch records into the bronze.batches_tracker table.

    This function scans the provided table metadata and creates tracking records
    for batches that have not been generated before. It prevents duplicate batch
    creation by checking existing records in the batch tracker.

    Workflow:
    1. Load existing batches from the tracker table.
    2. Check each new batch against previously generated batches.
    3. Skip batches that already exist (pending, failed, or optionally success).
    4. Create tracking records for new batches with a 'pending' status.
    5. Insert the new batch records into lake.bronze.batches_tracker.

    Args:
        tables (dict):
            Dictionary containing table configurations and generated batches.
            Expected format:
            {
                "customers": {
                    "new_batches": [
                        (1, 1, 1000),
                        (2, 1001, 2000)
                    ]
                }
            }

        load_type (str, optional):
            Type of load being processed (e.g., 'incremental', 'initial').
            This value is appended to the job name.
            Default is 'incremental'.

        tolerate_with_duplicate (bool, optional):
            If False (default), batches that were previously marked as
            'success' are skipped to avoid duplicate loading.

            If True, previously successful batches are ignored during the
            duplicate check, allowing the same data range to be generated
            again. This is mainly intended for full reloads or recovery
            scenarios where duplicate data insertion is acceptable.

    Returns:
        None
    """
    spark.catalog.clearCache()
    old_batches_q = "select * from lake.bronze.batches_tracker where  status in ('pending', 'failed' {})"
    if tolerate_with_duplicate:
        old_batches_q = old_batches_q.format("")
        
    else:
        old_batches_q = old_batches_q.format(", 'success' ")
    df_old_batches = spark.sql(old_batches_q).cache()
    data = []
    pbar = tqdm(tables,desc="Inserting New Batches", leave=True)
    for tp in pbar:
        
        job_name = tp +'_'+load_type
        pbar.set_postfix(table=tp, job_name= job_name)
        
        is_table_pending = df_old_batches.filter(F.col("job_name") ==  job_name)

        all_batches = tables.get(tp).get("new_batches")
        
        status = "pending"
        pbar2 = tqdm(all_batches,desc="all batches", leave=True)
        batch_df = spark.createDataFrame(
                        all_batches,
                        ["batch_number", "lower_bound", "upper_bound"]
                    )
        # same_prev_batch = (
        #                         df_old_batches
        #                         .join(
        #                             batch_df,
        #                             on=[
        #                                 df_old_batches.lower_bound == batch_df.lower_bound,
        #                                 df_old_batches.upper_bound == batch_df.upper_bound
        #                             ],
        #                             how="inner"
        #                         )
        #                     ).filter(F.col("job_name").contains(job_name))
        
        for batch_number, lower_bound, upper_bound in pbar2:
            pbar2.set_postfix(batch_number=batch_number, lower_bound= lower_bound, upper_bound= upper_bound)
            same_prev_batch = df_old_batches.filter(F.col("job_name").contains(tp) & 
                                                    ( ( F.col("lower_bound") == lower_bound) & 
                                                     ( F.col("upper_bound") == upper_bound)) )
            
            if  not same_prev_batch.isEmpty():
                status = same_prev_batch.select("status").distinct().collect()
                tqdm.write(f"this batch is exist in prev generateing for table {tp}, batch_number {batch_number} and there status: {status}")
                #tqdm.write_as_df(same_prev_batch)
                continue
            data.append((str(uuid.uuid4()), job_name, batch_number, lower_bound, upper_bound, status, None, None, None, None, datetime.now())) 
    df_new_batches = spark.createDataFrame(data, batch_tracker_schema)
    df_new_batches.createOrReplaceTempView("new_batch_tracker")
    spark.sql(""" 
                    INSERT INTO TABLE lake.bronze.batches_tracker 
                        SELECT * FROM new_batch_tracker;
                    """)
    
    df_old_batches.persist()
    tqdm.write("inserting successed")

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 59262)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/conda/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/usr/local/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/usr/local/spark/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/usr/local/spark/python/pyspark/accumulators.py", line 271, in accum_updates
    num_updates =

In [148]:
insert_new_batches(tables, load_type='inital_load', tolerate_with_duplicate = True)
print_as_df(spark.sql("select * from lake.bronze.batches_tracker;"))

In [589]:
df = spark.sql("select * from lake.bronze.batches_tracker where job_name like 'orders%' ")
print_as_df(df)

,id,job_name,batch_number,lower_bound,upper_bound,status,rows_processed,started_at_ts,completed_at_ts,error_message,generated_at_ts
0,e145ee88-5ec5-4435-96a7-d33aeeef2895,orders_inital_load,0,1,100000,success,100000,2026-06-09 21:47:52.727403,2026-06-09 21:47:57.922028,None,2026-06-09 21:47:16.394033
1,c2dbc2d3-17a0-4d83-83c0-23541dd100f6,orders_inital_load,1,100001,200000,success,100000,2026-06-09 21:48:00.974861,2026-06-09 21:48:06.086877,None,2026-06-09 21:47:16.421840
2,25a1bf7e-7ed3-4019-a149-72b1064aac11,orders_inital_load,0,1,100000,success,100000,2026-06-09 21:39:02.853076,2026-06-09 21:39:08.307563,None,2026-06-09 21:38:26.591680
3,25500d16-057f-4348-b9aa-4c9a8326995c,orders_inital_load,1,100001,200000,success,100000,2026-06-09 21:39:11.283716,2026-06-09 21:39:16.576186,None,2026-06-09 21:38:26.625182
4,b14e88c2-f692-45d0-b5b9-a9563f4a7112,orders_inital_load,2,200001,300000,success,100000,2026-06-09 21:39:19.648062,2026-06-09 21:39:24.829945,None,2026-06-09 21:38:26.657455


# Step 5 — Check for Existing Batches

# Step 6 — Process Each Batch

In [194]:
spark.sql(""" 
        CREATE TABLE IF NOT EXISTS lake.bronze.customers (
            operation         STRING,
            source_ts_ms      BIGINT,
            source_lsn        BIGINT,
            source_txid       BIGINT,
            source_table      STRING,
            source_snapshot   STRING,
            before_payload    STRING,
            after_payload     STRING,
            kafka_topic       STRING,
            kafka_partition   INT,
            kafka_offset      BIGINT,
            kafka_timestamp   TIMESTAMP,
            ingestion_ts      TIMESTAMP
        )
        USING iceberg
        PARTITIONED BY (days(ingestion_ts))
        TBLPROPERTIES (
            'format-version'                  = '2',
            'write.format.default'            = 'parquet',
            'write.parquet.compression-codec' = 'zstd',
            'write.target-file-size-bytes'    = '134217728'
        );

            """)
spark.sql(""" 
        CREATE TABLE IF NOT EXISTS lake.bronze.cities (
            operation         STRING,
            source_ts_ms      BIGINT,
            source_lsn        BIGINT,
            source_txid       BIGINT,
            source_table      STRING,
            source_snapshot   STRING,
            before_payload    STRING,
            after_payload     STRING,
            kafka_topic       STRING,
            kafka_partition   INT,
            kafka_offset      BIGINT,
            kafka_timestamp   TIMESTAMP,
            ingestion_ts      TIMESTAMP
        )
        USING iceberg
        PARTITIONED BY (days(ingestion_ts))
        TBLPROPERTIES (
            'format-version'                  = '2',
            'write.format.default'            = 'parquet',
            'write.parquet.compression-codec' = 'zstd',
            'write.target-file-size-bytes'    = '134217728'
        );

            """)
spark.sql(""" 
        CREATE TABLE IF NOT EXISTS lake.bronze.zones (
            operation         STRING,
            source_ts_ms      BIGINT,
            source_lsn        BIGINT,
            source_txid       BIGINT,
            source_table      STRING,
            source_snapshot   STRING,
            before_payload    STRING,
            after_payload     STRING,
            kafka_topic       STRING,
            kafka_partition   INT,
            kafka_offset      BIGINT,
            kafka_timestamp   TIMESTAMP,
            ingestion_ts      TIMESTAMP
        )
        USING iceberg
        PARTITIONED BY (days(ingestion_ts))
        TBLPROPERTIES (
            'format-version'                  = '2',
            'write.format.default'            = 'parquet',
            'write.parquet.compression-codec' = 'zstd',
            'write.target-file-size-bytes'    = '134217728'
        );

            """)
spark.sql(""" 
        CREATE TABLE IF NOT EXISTS lake.bronze.restaurants (
            operation         STRING,
            source_ts_ms      BIGINT,
            source_lsn        BIGINT,
            source_txid       BIGINT,
            source_table      STRING,
            source_snapshot   STRING,
            before_payload    STRING,
            after_payload     STRING,
            kafka_topic       STRING,
            kafka_partition   INT,
            kafka_offset      BIGINT,
            kafka_timestamp   TIMESTAMP,
            ingestion_ts      TIMESTAMP
        )
        USING iceberg
        PARTITIONED BY (days(ingestion_ts))
        TBLPROPERTIES (
            'format-version'                  = '2',
            'write.format.default'            = 'parquet',
            'write.parquet.compression-codec' = 'zstd',
            'write.target-file-size-bytes'    = '134217728'
        );

            """)

DataFrame[]

In [195]:
spark.sql("TRUNCATE TABLE lake.bronze.batches_tracker")

DataFrame[]

In [70]:
def mark_running(row_id:str):
        """
    Mark a batch as running in the batch tracker table.

    This function updates the status of a specific batch to `running`
    and records the batch start timestamp. After the update, it performs
    a validation step to ensure that the status change was applied
    successfully.

    Workflow:
    1. Update the batch status to `running`.
    2. Set `started_at_ts` to the current timestamp.
    3. Read the updated record from the tracker table.
    4. Verify that the batch exists.
    5. Verify that the status was correctly updated.
    6. Log the operation result.

    Args:
        row_id (str):
            Unique identifier of the batch record in
            `lake.bronze.batches_tracker`.

    Returns:
        None

    Raises:
        ValueError:
            - If the batch record does not exist.
            - If the status update was not applied successfully.

        AnalysisException:
            If the tracker table is missing or the SQL statement
            cannot be executed.

        Exception:
            Re-raises any unexpected errors after logging them.
    """
    try:
        spark.sql(f"""
            UPDATE lake.bronze.batches_tracker SET
                status = 'running',
                started_at_ts = CURRENT_TIMESTAMP
            WHERE id = '{row_id}'
        """)
        check = spark.sql(f"""
                SELECT status FROM lake.bronze.batches_tracker
                WHERE id = '{row_id}'
            """).collect()
        
        if not check:
            raise ValueError(f" Batch {row_id} not found in tracker")
        if check[0]["status"] != "running":
            raise ValueError(f"Failed to update batch {row_id} status")
        tqdm.write(f" status change to running. Successed for batch {row_id}")
    except AnalysisException as e:
        # Table missing or bad SQL — fatal
        tqdm.write(f" Fatal in mark_running func: tracker table issue — {e}")
        raise
    except Exception as e:
        # Connection or other — log and raise
        tqdm.write(f" Failed in mark_running func to mark batch {row_id} as running — {e}")
        raise

In [71]:
def mark_success(row_id:str, rows_processed:int):
     """
    Mark a batch as successfully completed in the batch tracker table.

    This function updates a batch record in `lake.bronze.batches_tracker`
    by setting its status to `success`, recording the completion timestamp,
    and storing the number of processed rows. It also validates that the
    update was applied correctly.

    Workflow:
    1. Update batch status to `success`.
    2. Set `completed_at_ts` to the current timestamp.
    3. Store the number of processed rows.
    4. Re-read the record from the tracker table.
    5. Validate existence of the batch.
    6. Confirm the status update was applied successfully.
    7. Log the successful completion.

    Args:
        row_id (str):
            Unique identifier of the batch record in
            `lake.bronze.batches_tracker`.

        rows_processed (int):
            Number of rows successfully processed in this batch.

    Returns:
        None

    Raises:
        ValueError:
            - If the batch record does not exist in the tracker table.
            - If the status update fails or is not reflected correctly.

        AnalysisException:
            If the tracker table is missing or the SQL update fails.

        Exception:
            Any unexpected error during execution is logged and re-raised.

    """
    try:
        spark.sql(f""" 
               UPDATE lake.bronze.batches_tracker SET
                   status = 'success', 
                   completed_at_ts = CURRENT_TIMESTAMP,
                   rows_processed = {rows_processed}   
               WHERE id = '{row_id}'
           """)
        check = spark.sql(f"""
                SELECT status FROM lake.bronze.batches_tracker
                WHERE id = '{row_id}'
            """).collect()

        if not check:
            raise ValueError(f"Batch {row_id} not found in tracker")
        if check[0]["status"] != "success":
            raise ValueError(f" Failed to update batch {row_id} status")
        tqdm.write(f" status change to success. Successed for batch {row_id} and row processed {rows_processed}")
    except AnalysisException as e:
       
        tqdm.write(f" Fatal in mark_success func: tracker table issue — {e} ")
        raise
    except Exception as e:
        tqdm.write(f" Failed in mark_success func to mark batch {row_id} as running — {e}")
        raise

In [149]:
def mark_failed(row_id:str, batch_number:str, error_msg:str):
    """
    Mark a batch as failed in the batch tracker table.

    This function updates the status of a batch in
    `lake.bronze.batches_tracker` to `failed`. It is typically used when
    a batch execution encounters an error. After updating the status,
    it validates that the update was successfully applied.

    Workflow:
    1. Update the batch status to `failed`.
    2. (Optionally) store or associate the failure reason.
    3. Re-read the batch record from the tracker table.
    4. Validate that the batch exists.
    5. Confirm that the status has been updated to `failed`.
    6. Log the failure event.

    Args:
        row_id (str):
            Unique identifier of the batch record in
            `lake.bronze.batches_tracker`.

        batch_number (str):
            Logical batch number used for tracking and logging purposes.

        error_msg (str):
            Error message describing the reason for the failure.

    Returns:
        None

    Raises:
        ValueError:
            - If the batch record does not exist in the tracker table.
            - If the status update is not reflected correctly.

        AnalysisException:
            If the tracker table is missing or the SQL statement fails.

        Exception:
            Any unexpected error is logged and re-raised.

    """
    try:
        spark.sql(f""" 
               UPDATE lake.bronze.batches_tracker SET
                   status = 'failed',
                   error_message = '{error_msg}'
                   
               WHERE id = '{row_id}'
                   
           """)
        check = spark.sql(f"""
                SELECT status FROM lake.bronze.batches_tracker
                WHERE id = '{row_id}'
            """).collect()

        if not check:
            raise ValueError(f"Batch {row_id} not found in tracker")
        if check[0]["status"] != "failed":
            raise ValueError(f" Failed to update batch {row_id} status")
        tqdm.write(f" status change to failed. Failed for batch {row_id} and batch number: {batch_number}")
    except AnalysisException as e:
       
        tqdm.write(f" Fatal in mark_failed func: tracker table issue — {e} ")
        raise
    except Exception as e:
        tqdm.write(f" Failed in mark_failed func to mark batch {row_id} as running — {e}")
        raise

In [150]:
def process_batches(tables:dict):
    """
    Execute batch processing for all tables using the batch tracker system.

    This function is the main orchestration layer for processing data
    batches in a Spark-based ingestion pipeline. It reads pending or
    failed batches from the batch tracker, extracts data from source
    tables using the defined batch ranges, transforms the data into a
    standardized ingestion format, and writes it to the target storage.

    Workflow:
    1. Iterate over all tables defined in the configuration.
    2. Fetch batches from `lake.bronze.batches_tracker` that are in
       `pending` or `failed` state.
    3. Skip tables that have no pending or failed batches.
    4. For each batch:
        a. Mark batch as `running`.
        b. Read data from the source table using batch bounds
           (lower_bound, upper_bound).
        c. Load data in parallel using partitioned reads.
        d. Add ingestion metadata columns (Kafka-style structure).
        e. Write transformed data to the target storage path.
        f. Mark batch as `success` with processed row count.
    5. If any error occurs:
        - Mark batch as `failed`.
        - Log the error and continue processing.

    Args:
        tables (dict):
            Dictionary containing table metadata and batch definitions.

            Example:
            {
                "customers": {
                    "PK": "customer_id",
                    "count": 41800,
                    "min_id": 1,
                    "max_id": 41800,
                    "new_batches": [
                        (0, 1, 41800)
                    ]
                }
            }

    Returns:
        None

    Side Effects:
        - Reads from source PostgreSQL tables.
        - Writes transformed data to target storage (e.g., Iceberg/Delta).
        - Updates batch status in `lake.bronze.batches_tracker`.

    Raises:
        None (all exceptions are handled per batch).

    Note:
        - Requires global variables:
            - `spark`
            - `bucket_name`
        - Requires helper functions:
            - `mark_running`
            - `mark_success`
            - `mark_failed`
            - `read_postgres_table_parallel`
        - Assumes each table has a valid primary key and batch ranges.
        - Designed for fault-tolerant batch reprocessing (retryable system).
    """
    pbar =  tqdm(list(tables.keys()), desc="Processing all tables", leave=True)
   # pbar.set_description(f"Processing all tables")
    for table in pbar:
        pbar.set_postfix(table=table)
        batch = spark.sql(f"""SELECT * 
                            FROM lake.bronze.batches_tracker 
                            WHERE job_name like '{table}%' and status in ('pending', 'failed') 
                            ORDER BY batch_number  """)
        if batch.isEmpty():
            tqdm.write(f"There is no batch in status pending or failed for table: {table}:")
           
            continue 
    
        batch.createOrReplaceTempView("updated_batches_tracker")
        
        #pbar2 = tqdm(batch.collect())
        for row in batch.collect():
           table_PK = tables.get(table).get("PK")
           path = f"{bucket_name}.{table}"
           tqdm.write(f" Start processing for table {table}: \n batch number:{row.batch_number}\n row id: {row.id}\n lower bound: {row.lower_bound}\n upper bound: {row.upper_bound}\n crrent status: {row.status} ")
           #tqdm.write(row.id, row.job_name, row.batch_number, row.lower_bound, row.upper_bound, row.status) 

           try:
               mark_running(row.id)
                
              
               general_q = f""" 
                          ( SELECT * 
                          FROM {table} as t 
                          WHERE {table_PK} >= '{row.lower_bound}'
                                  AND {table_PK} <= '{row.upper_bound}'
                          ) as q
                          """
               #tqdm.write(general_q)
               df = read_postgres_table_parallel(sql_query = general_q,
                                          partition_column= table_PK, 
                                          lower = row.lower_bound, 
                                          upper = row.upper_bound, 
                                          num_partitions=4)
               
               added_kafka_columns = df.select(
                                    F.lit("inital_load_"+table).alias("kafka_topic"),
                                    F.lit(None).alias("kafka_partition"),
                                    F.lit(None).alias("kafka_offset"),
                                    F.lit(None).alias("kafka_timestamp"),                 
                                    F.lit('r').alias("operation"),
                                    F.lit(None).alias("before_payload"),
                                    F.to_json(F.struct([F.col(c) for c in df.columns])).alias("after_payload"),
                                    F.lit(None).alias("source_ts_ms"),
                                    F.lit(1).alias("source_lsn"),
                                    F.lit(None).alias("source_txid"),
                                    F.lit(table).alias("source_table"),
                                    F.lit(table).alias("source_snapshot"),
                                    F.current_timestamp().alias("ingestion_ts")
                            )
               #tqdm.write_as_df(added_kafka_columns)
               rows_processed = df.count()
               tqdm.write(f" Number of rows processed {rows_processed}")
               tqdm.write(f" Path that will be weriting in: {path} ")
               (
                added_kafka_columns
                    .writeTo(path)
                    .append()
                )
               mark_success(row.id, rows_processed)
               tqdm.write("="*50)
           except Exception as e:
                mark_failed(row.id, row.batch_number, e)
                tqdm.write(e)
            
    

In [573]:
run_func(process_batches, tables)


Processing all tables:  55%|█████▍    | 6/11 [00:00<00:00, 24.02it/s, table=payments]   

There is no batch in status pending or failed for table: customers:
There is no batch in status pending or failed for table: cities:
There is no batch in status pending or failed for table: zones:
There is no batch in status pending or failed for table: restaurants:
There is no batch in status pending or failed for table: orders:
There is no batch in status pending or failed for table: order_items:


Processing all tables:  82%|████████▏ | 9/11 [00:00<00:00, 22.31it/s, table=menu_items]         

There is no batch in status pending or failed for table: payments:
There is no batch in status pending or failed for table: order_status_events:
There is no batch in status pending or failed for table: reviews:
There is no batch in status pending or failed for table: drivers:


Processing all tables: 100%|██████████| 11/11 [00:00<00:00, 22.23it/s, table=menu_items]

There is no batch in status pending or failed for table: menu_items:
Duration: 00:00:00.498


In [244]:

print_as_df(spark.sql(f""" select * 
                           from lake.bronze.batches_tracker where status ='success'
                           and job_name like 'customers%'
                    """), limit= 20)

,id,job_name,batch_number,lower_bound,upper_bound,status,rows_processed,started_at_ts,completed_at_ts,error_message,generated_at_ts
0,1ee08d4d-ea69-4de1-bf38-84363bd8839c,customers_inital_load,0,1,41801,success,20900,2026-06-08 23:59:00.054887,2026-06-08 23:59:03.731780,None,2026-06-08 23:58:54.099010
1,6e8e0f20-a95f-429e-9171-a49196ee7f4c,customers_inital_load,0,1,41801,success,20900,2026-06-08 21:25:00.808814,2026-06-08 21:25:03.957866,None,2026-06-08 21:24:45.907024
2,234ce2ff-f803-4962-8c3a-e1a04ddd318c,customers_inital_load,0,1,41801,success,20900,2026-06-08 23:06:59.876491,2026-06-08 23:07:03.361655,None,2026-06-08 23:06:53.928373


In [198]:
general_q = f"""
            ( SELECT * 
                          FROM customers as t 
                          ) as q
                          """
df = read_postgres_table_parallel(sql_query = general_q,
                                  partition_column= 'customer_id', 
                                  lower = 1, 
                                  upper = 10000, 
                                  num_partitions=4)
print_as_df(df)

,customer_id,email,full_name,phone,city_id,default_address,signup_date,is_active,created_at,updated_at
0,1,hassan.alsaadi.3844854@example.sa,Hassan Al-Saadi,+966528728463,2,"Jeddah, Saudi Arabia",2024-10-17,True,2026-05-12 20:07:56.477834,2026-05-12 20:07:56.477834
1,2,bader.alqurashi.633224@example.sa,Bader Al-Qurashi,+966513999315,5,"Dammam, Saudi Arabia",2026-03-01,True,2026-05-12 20:07:56.477834,2026-05-12 20:07:56.477834
2,3,turki.alruwais.7138374@example.sa,Turki Al-Ruwais,+966539587039,2,"Jeddah, Saudi Arabia",2025-05-28,True,2026-05-12 20:07:56.477834,2026-05-12 20:07:56.477834
3,4,salman.alqurashi.5808456@example.sa,Salman Al-Qurashi,+966547295260,3,"Mecca, Saudi Arabia",2026-01-12,True,2026-05-12 20:07:56.477834,2026-05-12 20:07:56.477834
4,5,waleed.almutairi.6122674@example.sa,Waleed Al-Mutairi,+966556164955,5,"Dammam, Saudi Arabia",2025-01-28,True,2026-05-12 20:07:56.477834,2026-05-12 20:07:56.477834


In [146]:
def run_pipeline(initial_load = False, is_initial_load_with_cdc = False, tolerate_with_duplicate = False):
    try:
        if initial_load:
            tables_with_details = get_metadata_tables(tables, initial_load = initial_load, is_initial_load_with_cdc = is_initial_load_with_cdc)
            
            load_type = 'inital_load'
        else:
            tables_with_details = get_metadata_tables(tables, initial_load = initial_load, is_initial_load_with_cdc = is_initial_load_with_cdc)
            load_type = 'incremntial_load'
     
        if not tables_with_details and load_type == 'incremntial_load':
            print(f"There is no updates on the tables\n {tables_with_details}")
        elif not tables_with_details and load_type == 'inital_load':
            print(f"There is no details on the tables\n {tables_with_details}")
        tables_with_batches = generate_batches(tables_with_details)
        insert_new_batches(tables_with_batches, load_type = load_type, tolerate_with_duplicate=tolerate_with_duplicate)
        process_batches(tables_with_batches)
    except Exception as e: 
        print("="*50)
        print("Error while runing the pipeline:\n", e)
        print("="*50)
run_func(run_pipeline, initial_load = True, is_initial_load_with_cdc = False, tolerate_with_duplicate = False)

Get Metadata Tables: 100%|██████████| 11/11 [00:00<00:00, 19385.44it/s]


technique type: inital load
table name:customers
 table PK:customer_id
0, table:customers, details:{'PK': 'customer_id', 'max_id': 41800, 'min_id': 1, 'count': 41800, 'is_cdc': False, 'new_batches': []}

technique type: inital load
table name:cities
 table PK:city_id
1, table:cities, details:{'PK': 'city_id', 'max_id': 5, 'min_id': 1, 'count': 5, 'is_cdc': False, 'new_batches': []}

technique type: inital load
table name:zones
 table PK:zone_id
2, table:zones, details:{'PK': 'zone_id', 'max_id': 61, 'min_id': 1, 'count': 61, 'is_cdc': False, 'new_batches': []}

technique type: inital load
table name:restaurants
 table PK:restaurant_id
3, table:restaurants, details:{'PK': 'restaurant_id', 'max_id': 3000, 'min_id': 1, 'count': 3000, 'is_cdc': False, 'new_batches': []}

table orders is cdc, reomved from the dict
table order_items is cdc, reomved from the dict
table payments is cdc, reomved from the dict
table order_status_events is cdc, reomved from the dict
table reviews is cdc, reomved 

Generate Batches: 100%|██████████| 4/4 [00:00<00:00, 337.31it/s, table=restaurants]


customers: batch_size_per_table 41800, count: 41800
cities: batch_size_per_table 5, count: 5
zones: batch_size_per_table 61, count: 61
restaurants: batch_size_per_table 3000, count: 3000


Inserting New Batches:  25%|██▌       | 1/4 [00:00<00:01,  2.97it/s, job_name=zones_inital_load, table=zones]        

this batch is exist in prev generateing for table customers, batch_number 0 and there status: [Row(status='success')]
this batch is exist in prev generateing for table cities, batch_number 0 and there status: [Row(status='success')]


Inserting New Batches: 100%|██████████| 4/4 [00:00<00:00,  6.04it/s, job_name=restaurants_inital_load, table=restaurants]


this batch is exist in prev generateing for table zones, batch_number 0 and there status: [Row(status='success')]
this batch is exist in prev generateing for table restaurants, batch_number 0 and there status: [Row(status='success')]
inserting successed


Processing all tables: 100%|██████████| 4/4 [00:00<00:00, 24.89it/s, table=restaurants]

There is no batch in status pending or failed for table: customers:
There is no batch in status pending or failed for table: cities:
There is no batch in status pending or failed for table: zones:
There is no batch in status pending or failed for table: restaurants:
Duration: 00:00:02.682


In [151]:
# df = spark.sql("select * from lake.bronze.customers where after_payload like '%customer_id%' ").filter(F.col("after_payload").contains('hassan.alsaadi.3844854') )
# print_as_df(df)
df = spark.sql("select * from lake.bronze.batches_tracker where job_name like 'order_status_events%'")
print_as_df(df)

,id,job_name,batch_number,lower_bound,upper_bound,status,rows_processed,started_at_ts,completed_at_ts,error_message,generated_at_ts
0,566c3683-b2a3-491c-9d6a-afb089aa1f8e,order_status_events_inital_load,0,1,100000,success,100000,2026-06-11 01:12:05.718624,2026-06-11 01:12:09.317941,None,2026-06-11 01:02:15.614697
1,4afcedce-fc5a-40f8-8216-86089c03e839,order_status_events_inital_load,1,100001,200000,success,100000,2026-06-11 01:12:11.250050,2026-06-11 01:12:14.823750,None,2026-06-11 01:02:15.626147
2,8ed7f05b-bfa3-4023-902f-9be1e5f8af67,order_status_events_inital_load,2,200001,300000,success,100000,2026-06-11 01:12:16.688867,2026-06-11 01:12:20.271223,None,2026-06-11 01:02:15.637656
3,c71b638c-1281-4c57-95d1-da80ec669a33,order_status_events_inital_load,3,300001,400000,success,100000,2026-06-11 01:12:22.301195,2026-06-11 01:12:26.109918,None,2026-06-11 01:02:15.648746
4,13293ce4-4870-4fad-bac9-e2c194e8d4e9,order_status_events_inital_load,4,400001,500000,success,100000,2026-06-11 01:12:28.072062,2026-06-11 01:12:31.555737,None,2026-06-11 01:02:15.660320
